# Assumption

The README did not specify if we are to use the SMTLib distribution for Z3. I am hence assuming that we are to use the python API [(documentation)](https://ericpony.github.io/z3py-tutorial/guide-examples.htm).

For me, it is easier to parse and has a cleaner syntax. I also used it for Assignment 1 and so have some level of familiarity with it. Python with jupyter notebooks also allows for easier writeups with code.



### Imports

In [260]:
from z3 import Solver, Bool, And, Not, Or, Ints, Int, Implies, ForAll, Reals, sat, Sum, Optimize, Real, If
from sympy import symbols, sympify
from IPython.display import Markdown
import numpy as np

### **Game of 21**

We can recursively understand the logic behind the model. 

- Firstly, note that a player who is left with a pile of size 1, 2, or 3, can always win.
    - Thus, $Win(1) = Win(2) = Win(3) = \top$
        - This is the base case, kind of like axioms

- Next, note that if a player has a winning position, it means that there is some removal choice they can make so that the resulting configation is not a winning postion. In other words, having a pile of size either 1 object, 2 objects or 3 objects lesser than the current pile size that leads to a loss.
    - So, $Win(i) = \neg Win(i-1) \lor \neg Win(i-2) \lor \neg Win(i-3)$
        - This is the recursive description, kind of like a rule of inference
        - $1 < n \leq 21$
        - The winning strategy would be to always pick this removal choice.

In [261]:
s = Solver()
Win = [Bool(f"Win({i})") for i in range(1,22)]

In [262]:
# base case
s.add(Win[0]==True, Win[1]==True, Win[2]==True)
# indexing is weird

s

[Win(1) == True, Win(2) == True, Win(3) == True]

In [263]:
# recursion
for i in range(3,21): # adjustment for zero indexing
    s.add(Win[i] == Or(Not(Win[i-1]), Not(Win[i-2]), Not(Win[i-3])))

s.push()
s

[Win(1) == True,
 Win(2) == True,
 Win(3) == True,
 Win(4) == Or(Not(Win(3)), Not(Win(2)), Not(Win(1))),
 Win(5) == Or(Not(Win(4)), Not(Win(3)), Not(Win(2))),
 Win(6) == Or(Not(Win(5)), Not(Win(4)), Not(Win(3))),
 Win(7) == Or(Not(Win(6)), Not(Win(5)), Not(Win(4))),
 Win(8) == Or(Not(Win(7)), Not(Win(6)), Not(Win(5))),
 Win(9) == Or(Not(Win(8)), Not(Win(7)), Not(Win(6))),
 Win(10) == Or(Not(Win(9)), Not(Win(8)), Not(Win(7))),
 Win(11) == Or(Not(Win(10)), Not(Win(9)), Not(Win(8))),
 Win(12) == Or(Not(Win(11)), Not(Win(10)), Not(Win(9))),
 Win(13) == Or(Not(Win(12)), Not(Win(11)), Not(Win(10))),
 Win(14) == Or(Not(Win(13)), Not(Win(12)), Not(Win(11))),
 Win(15) == Or(Not(Win(14)), Not(Win(13)), Not(Win(12))),
 Win(16) == Or(Not(Win(15)), Not(Win(14)), Not(Win(13))),
 Win(17) == Or(Not(Win(16)), Not(Win(15)), Not(Win(14))),
 Win(18) == Or(Not(Win(17)), Not(Win(16)), Not(Win(15))),
 Win(19) == Or(Not(Win(18)), Not(Win(17)), Not(Win(16))),
 Win(20) == Or(Not(Win(19)), Not(Win(18)), Not(Win(17))),
 Win(21) == Or(Not(Win(20)), Not(Win(19)), Not(Win(18)))]

- Finally, we check if the first player can win, knowing that they always start with a pile size of 21.
    - We want to show that the above setup leads to the conclusion $Win(21)$.
    - We add the negation of the conclusion to the premises and show that it is UNSAT.

In [264]:
s.add(Win[20] == False) # 20 because indexing

s.check()

unsat

The first player can thus always win. What about the winning strategy?

- Let's see if we can infer something by pushing $Win(21) = \top$ and getting the correponding valualtion (model)

In [265]:
s.pop()

s.add(Win[20] == True)

display(s.check())
s.model()

sat

[Win(1) = True,
 Win(4) = False,
 Win(9) = True,
 Win(17) = True,
 Win(13) = True,
 Win(20) = False,
 Win(14) = True,
 Win(15) = True,
 Win(21) = True,
 Win(3) = True,
 Win(16) = False,
 Win(18) = True,
 Win(19) = True,
 Win(5) = True,
 Win(8) = False,
 Win(10) = True,
 Win(6) = True,
 Win(11) = True,
 Win(12) = False,
 Win(7) = True,
 Win(2) = True]

- Let's observe the model more carefully:

In [266]:
for i in range(20,-1,-1):
    display(Markdown(f"For pile of size = {i+1}, can we make a winning choice, $ Win({i+1})$? $ \\ \\ {s.model()[Win[i]]}$"))

For pile of size = 21, can we make a winning choice, $ Win(21)$? $ \ \ True$

For pile of size = 20, can we make a winning choice, $ Win(20)$? $ \ \ False$

For pile of size = 19, can we make a winning choice, $ Win(19)$? $ \ \ True$

For pile of size = 18, can we make a winning choice, $ Win(18)$? $ \ \ True$

For pile of size = 17, can we make a winning choice, $ Win(17)$? $ \ \ True$

...

For pile of size = 5, can we make a winning choice, $ Win(5)$? $ \ \ True$

For pile of size = 4, can we make a winning choice, $ Win(4)$? $ \ \ False$

For pile of size = 3, can we make a winning choice, $ Win(3)$? $ \ \ True$

For pile of size = 2, can we make a winning choice, $ Win(2)$? $ \ \ True$

For pile of size = 1, can we make a winning choice, $ Win(1)$? $ \ \ True$

- Note that every multiple of 4 is false. This is precisesly the winning strategy. 


    - The first player can always force the second player into a multiple of 4. 
    - The second player can never force the first player into a multiple of 4.
    - Assuming optimal play by the first player, eventually, the second player will be forced into a state with a file size of 4.
        - They cannot remove all object from the pile here, so they remove 1, 2, or 3 onjects. 
        - Whatever their choice, the first player ends up in cases they necessarily win!

### **Non Linear Constraint Solving**

This is pretty straightforward, we just load the statements into a z3 solver.

In [267]:
x, y = Ints('x y')

In [268]:
type(x)

z3.z3.ArithRef

In [269]:
s = Solver()

In [270]:
s.add(
    x*x + y*y == 25,  # pyright: ignore[reportOperatorIssue]
    x + y == 7,
    x > 0,
    y > 0
)

s

[x*x + y*y == 25, x + y == 7, x > 0, y > 0]

In [271]:
s.check()

sat

While SAT, iterate append negation of solution to find new assignment, till UNSAT is reached. Display each solution as it comes.

In [272]:
while str(s.check()) != 'unsat':

    m = s.model()

    sol_x = int(str(m[x]))
    sol_y = int(str(m[y]))

    print("x =", sol_x, ", y =", sol_y)
    s.add(Not(And(x == sol_x, y == sol_y)))

x = 3 , y = 4
x = 4 , y = 3


$(x=3, y=4)$ and $(x=4,y=3)$ are the only integer solutions.

### **(Inductive) Invariant Synthesis Example**

- At the time of writing - it's almost the end of the semester and I'm running short on time. So I'm going to outline an approach that we'll reuse in Week 6 (and subsequently) for invariant synthesis
 
- We'll implement Farkas' Lemma as described in [\[CVA03.pdf\]](..\week8\papers\CAV03.pdf) - for any d variables in a program. In the invariant, these variables may be products of each other. Let's formulate mathematically.

We have program variables $v = \begin{bmatrix}x_1 \\ x_2 \\ \vdots \\ x_d \end{bmatrix}$
We construct the following basis over the program variables:

$$
b(v) = \begin{bmatrix}1 \\ x_1 \\ x_2 \\ \vdots \\ x_d \\ x_1x_2 \\ \vdots \\ x_2^2 \\ \vdots \\ x_1^3 \\ \vdots \\ x_1x_2x_3...x_d \\ \vdots \\ x_d^d\end{bmatrix}
$$

Our Linear\* Invariant(s) will be over this basis. Consider the coefficients:

$$c = \begin{bmatrix}a_1 \\ a_2 \\ \vdots \\ a_k\end{bmatrix}$$

\*These are linear not in $v$ but in $B$. Also, $k$ is at most $\binom{2d}{d}$

The expression for our invariant inequality becomes

$$I(b(v)) = c^{\top} \times b(v) \leq 0$$

For our base case, let $v_{init}$ be the initial state of the loop variables, $b(v)_{init}$ is the initial state of the basis variables. We ensure that $I(b(v)_{init}) \leq 0$


Now, we have three types of constraints:

1. Assignment Constraints 

    `x := x+1`

2. Condition Constraints 

    `while (i<n)`

3. Loop Body Constraints 

    `if (x > 0) {y := y + 1;}`

Note that 1 and 2 can be combined to give something like 3, which holds within the loop. We can thus write any expression in the loop in this format. 

In other words, a given path through a loop thus contains conditions or 'guards' on the variables, $G(v)$, and assignments or transitions on the variables, $T(v)$. We write:

$$
I(b(v)) \land G(b(v)) \implies I(b(T(v)))
$$

Let's formalise $G(b(v))$. Suppose we have some execution path through the loop. We set some guard coefficients

$$
g = \begin{bmatrix}g_1 \\ g_2 \\ \vdots \\ g_k \end{bmatrix}
$$

And note that

$$
G(b(v)) = g^\top \times b(v) \leq 0
$$

This inequality on the basis describes the while loop condition and the respective execution path it takes through if conditions (if present).

Similarly, let's formalise $b(T(v))$. $T(v)$ defines a vector containing the transition updates of every variable. We update everything in the basis with its transition value and expand out.

This gives us $b(T(v))$, a new basis with linear combination of the original basis elements at each entry.

We know that this can be represented with a matrix multiplication! Consider a $k \times k$ square matrix $M$ where each row contains the coefficients of the transition state for the corresponding entry in the original basis. 

There are k columns, each with the coefficient of a particular basis element in the transition state of that entry.

To help visualise - suppose the $i$ th entry of $T(v)$, $T(v)_i$, is given by $t_0x_1 + t_1x_2 + \dots + t_dx_d$, i.e. the transition update for variable $x_i$. Note that this is always  in $v$ - see next point for why.

Applying these transitions to some $b(v)_j$, we expand out the terms and end up with a new linear combination over the variables of $b(v)$ This is $b(T(v))_j = \begin{bmatrix}b_{j1} & b_{j2} & b_{j3} & \dots & b_{jk} \end{bmatrix} b(v)$.

If $T(v)$ was not linear in $v$, than this basis would no longer be a linear combination of the variables in $b(v)$ - it would be a new basis with a new set of terms, which would keep growing for each update!

Anyway, row $j$ of the matrix $M$ will have the column entries $b_{j1}, b_{j2} , b_{j3} , \dots , b_{jk}$.

We can thus write:

$$
b(T(v)) = M \times b(v)
$$

(This will be clearer with an example later on.)

Hence, we can derive the following:

$$
I(b(v)) \land G(b(v)) \implies I(b(T(v)))
$$

$$
(c^\top b(v) \leq 0) \land (g^\top b(v) \leq 0) \implies (c^\top  M b(v) \leq 0)
$$

By [Farkas' Lemma](https://www.mat.univie.ac.at/~rabot/publications/jour05-01.pdf), this implication holds **iff** $\exists \ \lambda_0,\lambda_1 \geq 0$ such that:

$$
c^\top  M b(v) = \lambda_0 c^\top b(v) + \lambda_1 g^\top b(v)
$$

Now,

$$
c^\top  M b(v) = \lambda_0 c^\top b(v) + \lambda_1 g^\top b(v)
$$

$$
\implies (c^\top  M b(v))^\top = (\lambda_0 c^\top b(v))^\top + (\lambda_1 g^\top b(v))^\top
$$

$$
\implies (c^\top  M b(v))^\top = \lambda_0 (c^\top b(v))^\top + \lambda_1 (g^\top b(v))^\top
$$

$$
\implies b(v)^\top M^\top c = \lambda_0 b(v)^\top c  + \lambda_1 b(v)^\top g 
$$

$$
\implies b(v)^\top M^\top c =  b(v)^\top \lambda_0 c  +  b(v)^\top \lambda_1 g 
$$

$$
\implies M^\top c = \lambda_0  c +  \lambda_1 g 
$$

We just reduced our inductive invariant to a system of equations, which, as noted in the previous section, can be solved using Non-Linear Constraint Solving in Z3! 

Our unknowns are the real coefficients $c$, and scalars $\lambda_0,\lambda_1$. The scalars must be $\geq 0$. We also add the constraint that $c^\top c = 1$ to normalize the coefficients - this ensures that the solver doesn't keep coming up with scalar multiples of the same expression.

It's time to run through this using `i` and `s` (`s` representing `sum`)

In [273]:
#i = int()
#s = int()
n = 10

#v = [i, s]
#b_v = [1, i, s, i*i, i*s, s*s]

In [274]:
M = np.array([
        [1, 0, 0, 0, 0, 0], # 1 = 1
        [1, 1, 0, 0, 0, 0], # i = i+1 = 1*1 + 1*i + 0*(others)
        [0, 1, 1, 0, 0, 0], # s = s+i = 0*1 + 1*i + 1*s ...etc
        [1, 2, 0, 1, 0, 0], # (i+1)^2 = i^2 + 2i + 1
        [0, 1, 1, 1, 1, 0], # (i+1)(s+i) = i + s + i^2 + i*s
        [0, 0, 0, 1, 2, 1], # (s+i)^2 = i^2 + 2*i*s + s^2 
    ])

M.T

array([[1, 1, 0, 1, 0, 0],
       [0, 1, 1, 2, 1, 0],
       [0, 0, 1, 0, 1, 0],
       [0, 0, 0, 1, 1, 1],
       [0, 0, 0, 0, 1, 2],
       [0, 0, 0, 0, 0, 1]])

In [275]:
g = np.array([1-n, 1, 0, 0, 0, 0])# i < n ===> i <= n-1 ===> 1*(1-n) + 1*i + 0*(others)

g

array([-9,  1,  0,  0,  0,  0])

In [276]:
c = np.array((Reals('a1 a2 a3 a4 a5 a6')))

c

array([a1, a2, a3, a4, a5, a6], dtype=object)

In [277]:
λ0, λ1 = Reals('λ0 λ1')

In [278]:
lhs = M.T @ c
rhs = c*λ0 + g*λ1

lhs, rhs

(array([1*a1 + 1*a2 + 0*a3 + 1*a4 + 0*a5 + 0*a6,
        0*a1 + 1*a2 + 1*a3 + 2*a4 + 1*a5 + 0*a6,
        0*a1 + 0*a2 + 1*a3 + 0*a4 + 1*a5 + 0*a6,
        0*a1 + 0*a2 + 0*a3 + 1*a4 + 1*a5 + 1*a6,
        0*a1 + 0*a2 + 0*a3 + 0*a4 + 1*a5 + 2*a6,
        0*a1 + 0*a2 + 0*a3 + 0*a4 + 0*a5 + 1*a6], dtype=object),
 array([a1*λ0 + -9*λ1, a2*λ0 + 1*λ1, a3*λ0 + 0*λ1, a4*λ0 + 0*λ1,
        a5*λ0 + 0*λ1, a6*λ0 + 0*λ1], dtype=object))

In [279]:
farkas_constraints = [lhs[i] == rhs[i] for i in range(len(c))]

farkas_constraints

[1*a1 + 1*a2 + 0*a3 + 1*a4 + 0*a5 + 0*a6 == a1*λ0 + -9*λ1,
 0*a1 + 1*a2 + 1*a3 + 2*a4 + 1*a5 + 0*a6 == a2*λ0 + 1*λ1,
 0*a1 + 0*a2 + 1*a3 + 0*a4 + 1*a5 + 0*a6 == a3*λ0 + 0*λ1,
 0*a1 + 0*a2 + 0*a3 + 1*a4 + 1*a5 + 1*a6 == a4*λ0 + 0*λ1,
 0*a1 + 0*a2 + 0*a3 + 0*a4 + 1*a5 + 2*a6 == a5*λ0 + 0*λ1,
 0*a1 + 0*a2 + 0*a3 + 0*a4 + 0*a5 + 1*a6 == a6*λ0 + 0*λ1]